# Анализ аудитории витуберов

Посмотрим ключевую статистику комьюнити витуберов:

**Что сделано**:

* Рост аудитории
* Уникальность витуберов
* Кластеризация витуберов
* Количество локальных фолловов у зрителей выборки
* Количество глобальных фолловов у зрителей выборки
* Граф самых близости витуберов (то есть если косинусное расстояние меньше какого-то порога, то рисовать ребро, иначе - нет)

**Что пока нет**:

* Размер активной аудитории
* Активность чата
* Кластеризация сообщений в чате

#### Импорт всего нужного

In [2]:
import os
import csv
import sys
sys.path.insert(1, '../util/')

from data import UserData, FollowerData, json_to_user_data
from userdata import get_userdata, get_userdata_by_login
from followers import get_followers

import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd
import numpy as np
import sklearn
from datetime import datetime, timedelta
from typing import Optional, List, Tuple, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

## Prepare data

In [3]:
VTUBERS_LIST_FILE_PATH = "./data/vtubers.txt"

In [4]:
@dataclass
class VtubersData:
    vtuber_names: List[str] # 1 x N
    vtuber_followers: Dict[str, List[FollowerData]]
    user_following: Dict[int, List[str]]

### Check vtuber names:

In [ ]:
filelines = []
vtuber_names = set()
vtuber_ids = set()

with open(VTUBERS_LIST_FILE_PATH, 'r') as data_file:
    counter = 0
    
    for fileline in data_file.readlines():
        counter += 1
        print(f"Reading line #{counter}")
        fileline = fileline.strip()
        if '#' in fileline:
            line, comment = fileline.split('#', 1)
        else:
            line, comment = fileline, ""
        
        line = line.strip()
        if line:  # Got vtuber name or vtuber id
            try:
                vtuber_id = int(line)
                vtuber_name = None
            except ValueError:
                vtuber_id = None
                vtuber_name = line
            
            if vtuber_id is not None:
                if vtuber_id in vtuber_ids:
                    print("WARNING: duplicate vtuber found:", fileline)
                    continue

                vtuber_data = get_userdata(id=vtuber_id, log=True)
                
                if vtuber_data is None:
                    print(f"WARNING: no vtuber with id {vtuber_id} found:", fileline)
                    filelines.append(fileline)
                else:
                    if comment.strip().lower() != vtuber_data.login.lower():
                        new_fileline = f"{vtuber_data.id} # {vtuber_data.login.lower()}"
                        print("WARNING: should update line:", fileline, "-->", new_fileline)
                        filelines.append(new_fileline)
                    else:
                        filelines.append(fileline)
                    vtuber_names.add(vtuber_data.login)
                    vtuber_ids.add(vtuber_data.id)
            else:
                if vtuber_name in vtuber_names:
                    print("WARNING: duplicate vtuber found:", fileline)
                    continue
                vtuber_data = get_userdata_by_login(login=vtuber_name, log=True)

                if vtuber_data is None:
                    print(f"WARNING: no vtuber with name {vtuber_name} found:", fileline)
                    filelines.append(fileline)
                else:
                    new_fileline = f"{vtuber_data.id} # {vtuber_name}"
                    print("WARNING: should update line:", fileline, "-->", new_fileline)
                    vtuber_names.add(vtuber_data.login)
                    vtuber_ids.add(vtuber_data.id)
                    filelines.append(new_fileline)
        else: # Got empty line or comment line
            filelines.append(fileline)

vtuber_names = list(vtuber_names)

In [ ]:
# Run if you think you should update vtubers.txt file
with open(VTUBERS_LIST_FILE_PATH, 'w') as data_file:
    for fileline in filelines:
        data_file.write(fileline)
        data_file.write("\n")

### Load vtuber names without checking

In [5]:
vtuber_names = set()

with open(VTUBERS_LIST_FILE_PATH, 'r') as data_file:
    for fileline in data_file.readlines():
        fileline = fileline.strip()
        if '#' in fileline:
            line, comment = fileline.split('#', 1)
            line = line.strip()
            comment = comment.strip()
        else:
            line, comment = fileline, ""
            line = line.strip()

        if line:
            try:
                vtuber_id = int(line)
                vtuber_name = comment
            except ValueError:
                vtuber_id = None
                vtuber_name = line
            
            if vtuber_name is not None:
                vtuber_names.add(vtuber_name.lower())

vtuber_names = list(vtuber_names)

In [6]:
print(f"Loaded {len(vtuber_names)} vtubers")

Loaded 750 vtubers


### Load followers data from cache or gql

In [7]:
def get_cache_path():
    current_date = datetime.now()

    return os.path.join("data/", 
                        ".temp/",
                        f"followers_{current_date.year}_{current_date.month}_{current_date.day}.csv")


def load_followers_data(vtuber_names: List[str],
                        online: bool = False, 
                        num_workers: int = 32,
                        cache_path: str = get_cache_path()):
    if online:
        def load_followers_execution(vtuber_login: str) -> Tuple[str, Optional[List[FollowerData]]]:
            result = get_followers(vtuber_login, log=False, repeat_times=10, repeat_delay=1.0)
            if result is None:
                print("[ERROR]", "Failed loading vtuber", vtuber_login)
                return vtuber_login, None
            else:
                return vtuber_login, result


        vtuber_followers = {}
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = []

            for vtuber in vtuber_names:
                future = executor.submit(load_followers_execution, vtuber_login=vtuber)
                futures.append(future)
            
            for future in as_completed(futures):
                vtuber_login, followers_result = future.result()
                if followers_result is None:
                    print(f"Cannot load `{vtuber_login}` result")
                    followers_result = []
                else:
                    empty_results = list(filter(lambda el: el.user is None, followers_result))
                    print(f"Got `{vtuber_login}` result, empty results: {len(empty_results)}/{len(followers_result)}")
                vtuber_followers[vtuber_login] = followers_result
        
        return vtuber_followers
    else:
        result = {}

        with open(cache_path, 'r') as cache_file:
            csvreader = csv.DictReader(cache_file, delimiter=',')

            for row in csvreader:
                vtuber = row['vtuber']
                followed_at = datetime.fromisoformat(row['followed_at'])
                if row['id']:
                    id              = int(row['id'])
                    login           = None if row['login'] == "None" else row['login']
                    created_at      = None if row['created_at'] == "None" else datetime.fromisoformat(row['created_at'])
                    deleted_at      = None if row['deleted_at'] == "None" else datetime.fromisoformat(row['deleted_at'])
                    follows_count   = int(row['follows_count'])
                    user_data = UserData(id=id,
                                        login=login,
                                        created_at=created_at,
                                        deleted_at=deleted_at,
                                        follows_count=follows_count)
                else:
                    user_data = None
                follower_data = FollowerData(user=user_data,
                                            followed_at=followed_at)
                
                if vtuber not in result.keys():
                    result[vtuber] = []
                result[vtuber].append(follower_data)
        
        return result


def cache_followers_data(vtuber_followers: Dict[str, List[FollowerData]],
                        cache_path: str = get_cache_path()):
    with open(cache_path, 'w') as cache_file:
        csvwriter = csv.writer(cache_file, delimiter=',')
        csvwriter.writerow(["vtuber",
                            "followed_at",
                            "id",
                            "login",
                            "created_at",
                            "deleted_at",
                            "follows_count"])

        for (vtuber, value) in vtuber_followers.items():
            for follower_data in value:
                if follower_data.user is None:
                    csvwriter.writerow([vtuber,
                                        follower_data.followed_at,
                                        "",
                                        "",
                                        "",
                                        "",
                                        ""])
                else:
                    csvwriter.writerow([vtuber,
                                        follower_data.followed_at,
                                        follower_data.user.id,
                                        follower_data.user.login,
                                        str(follower_data.user.created_at),
                                        str(follower_data.user.deleted_at),
                                        follower_data.user.follows_count])

In [8]:
# vtuber_followers = load_followers_data(None, online=False, cache_path="./data/.temp/followers_2026_3_2.csv")
vtuber_followers = load_followers_data(None, online=False, cache_path="./data/.temp/followers_2026_6_15.csv")
# vtuber_followers = load_followers_data(vtuber_names, online=True, num_workers=64)

In [9]:
# IGNORED_VTUBERS = ['xrogozik', 'd_maf_', 'blin_ya', 'squimeh', 'solemuri', 'miuchia', 'ellysteiny', 'maroosha', 'simfonira', 'qchaan_9', 'banshameow', 'nyamuras', 'yuislime', 'rera_seal', 'xkamysh', 'sleduck', 'lerritay', 'trixie_vox', 'dear_hellgirl']
IGNORED_VTUBERS = []

In [10]:
vtuber_names = list(vtuber_followers.keys())
vtuber_names

['nesaga_micune',
 'fumm_y',
 'memumemumemumemu',
 'cielviva',
 'novachris_',
 'mmeoowee',
 'sei_tenko',
 'ofion_',
 'nameless_lis',
 'murmurs_',
 'meow_toki',
 'fey_ling',
 'rinashivt',
 'mahiruchan',
 'zxc_k0t',
 'fatui_tyan',
 'plurrrentia',
 'lunifox__',
 '0_0nari_chan0_0',
 'murakatatsu',
 'dao_show',
 'd_e_l_y',
 'vvurst_exe',
 'milloyure',
 'baxieoff',
 'linda_medic',
 'pepesilvia669',
 'ironbell_tv',
 'sheeppyoz',
 'farostg',
 'rom4shka_uwu',
 'rua_sato',
 'mrden1l',
 'fair_soul',
 'bl00ming',
 'glumarkoj',
 'mon_misa',
 'katsumi__chan',
 'wakuforia',
 'oxyfoxys',
 'th4_vo1dtwink',
 'hebihidden',
 'kwakusha228',
 'marffa_',
 'sladkayadoza69',
 'raychivt',
 'blyoka',
 'psyhessa',
 'aoiwer_uwu',
 'felixkohai',
 'kroliksudby',
 'yukkithejaba',
 'charlotte__ch',
 'dobraya88',
 'hurmyasha',
 'estishe4ka',
 'chiiko_chan',
 'mionavt',
 'etto_teto',
 'silverosemary',
 'magicfoxg',
 'pineapplevt',
 'walha11er',
 'ninora_art',
 'vampy_art',
 'hideria',
 'bumajk4',
 'ai_missouri',
 'kasht

In [11]:
for vtuber in vtuber_names:
    if vtuber_followers[vtuber] is None or len(vtuber_followers[vtuber]) == 0:
        print(vtuber)

In [12]:
user_following = {}
vtuber_names_with_ignored = []

for vtuber in vtuber_names:
    if vtuber in IGNORED_VTUBERS:
        print("Ignored vtuber:", vtuber)
        continue
    vtuber_names_with_ignored.append(vtuber)
    for follower in vtuber_followers[vtuber]:
        if follower.user is not None:
            user_name = follower.user.id
            if user_name not in user_following.keys():
                user_following[user_name] = []
            user_following[user_name].append(vtuber)

In [13]:
data = VtubersData(vtuber_names=vtuber_names_with_ignored, vtuber_followers=vtuber_followers, user_following=user_following)

### Cache vtuber followers data

In [ ]:
cache_followers_data(vtuber_followers)

## Графики роста аудитории

### График фолловов стримера

In [ ]:
import plotly.io as pio

pio.templates.default = 'plotly_dark'

pio.renderers

In [ ]:
sunce_date = datetime.fromisoformat("2024-01-01 00:00:00.000000+00:00")
STREAMER = 'xkamysh'
auditory_data = data.vtuber_followers[STREAMER]
auditory_data = list(map(lambda viewer: viewer.followed_at, auditory_data))
auditory_data = list(filter(lambda follow_date: follow_date > sunce_date, auditory_data))
auditory_data = np.array(auditory_data)


df = pd.DataFrame({
    "x": auditory_data,
})
fig = px.histogram(df,
                   title=f"График фолловеров в день у {STREAMER} начиная с {sunce_date}",
                   nbins=1000,
                   x="x",
                   labels={"x": "Date"})
fig.show()

### График фолловов

In [ ]:
len(data.vtuber_names), len(data.user_following), sum(map(len, data.user_following.values()))

In [ ]:
sunce_date = datetime.fromisoformat("2024-01-01 00:00:00.000000+00:00")
auditory_data = []
for vtuber in data.vtuber_names:
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        auditory_data.append(follower)

auditory_data = list(map(lambda viewer: viewer.followed_at, auditory_data))
auditory_data = list(filter(lambda follow_date: follow_date > sunce_date, auditory_data))
auditory_data = np.array(auditory_data)


df = pd.DataFrame({
    "x": auditory_data,
})
fig = px.histogram(df,
                   title="Количество фолловеров за промежуток",
                   x="x",
                   range_y=(0, 40000))
fig.show()

### График фолловеров без причин резких скачков

In [ ]:
sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = []
for vtuber in data.vtuber_names:
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        auditory_data.append(follower)

auditory_data = list(map(lambda viewer: viewer.followed_at, auditory_data))
auditory_data = list(filter(lambda follow_date: follow_date > sunce_date, auditory_data))
auditory_data = np.array(auditory_data)

df = pd.DataFrame({
    "x": auditory_data,
})
fig = px.histogram(df,
                   title=f"Рост аудитории с {sunce_date}",
                   x="x",
                   cumulative=True)
# fig.write_image(f"followers_since_{sunce_date}_cumulative.png", width=1980, height=1080)
fig.show() 

Глянул, кто подписывался на кого в конкретный день:

In [ ]:
watching_date = datetime(2024, 10, 17)

watching_followers = []
watching_followers_vtubers = {}
for vtuber in data.vtuber_names:
    if vtuber in IGNORED_VTUBERS:
        continue
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        if follower.followed_at.year == watching_date.year and \
                follower.followed_at.month == watching_date.month and \
                follower.followed_at.day == watching_date.day:
            watching_followers.append((vtuber, follower))
            watching_followers_vtubers[vtuber] = watching_followers_vtubers.get(vtuber, 0) + 1

In [ ]:
list(sorted(watching_followers_vtubers.items(), key=lambda el: el[1], reverse=True))[:10]

### Рост уникальных фолловов

То есть не учитываются второй, третий и т.д. фолловы от одного и того же человека

In [ ]:
sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = {}
for vtuber in data.vtuber_names:
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        if follower.user is not None:
            if follower.user.login not in auditory_data.keys():
                auditory_data[follower.user.login] = follower.followed_at
            auditory_data[follower.user.login] = min(auditory_data[follower.user.login], follower.followed_at)

auditory_data = auditory_data.values()
auditory_data = list(filter(lambda follow_date: follow_date > sunce_date, auditory_data))
auditory_data = np.array(auditory_data)

df = pd.DataFrame({
    "x": auditory_data,
})
fig = px.histogram(df, 
                   title=f"Рост уникальных фолловов начиная с {sunce_date}",
                   x="x",
                   cumulative=True)
fig.show() 

## Живая аудитория

In [ ]:
# Активная аудитория - которая подписана хотя бы на двух витуберов
active_auditory = set()
for vtuber in data.vtuber_names:
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None and len(data.user_following[follower.user.id]) >= 2:
            active_auditory.add(follower.user.id)

auditory_data = {}
for vtuber in data.vtuber_names:
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        if follower.user is not None and follower.user.id in active_auditory:
            if follower.user.login not in auditory_data.keys():
                auditory_data[follower.user.login] = follower.followed_at
            auditory_data[follower.user.login] = min(auditory_data[follower.user.login], follower.followed_at)

sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
until_date = datetime.fromisoformat("2025-07-01 00:00:00.000000+00:00")
auditory_data = auditory_data.values()
auditory_data = list(filter(lambda follow_date: follow_date > sunce_date, auditory_data))
auditory_data = list(filter(lambda follow_date: follow_date < until_date, auditory_data))
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = np.array(auditory_data)

df = pd.DataFrame({
    "x": auditory_data,
})
fig = px.histogram(df, x="x")
fig.show() 

In [ ]:
len(auditory_data)

Если человек подписыывается на хотя бы двух, то он подписывается на второго достаточно быстро?

In [ ]:
user_follow_dates = {}
since_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
for vtuber in data.vtuber_names:
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None:
            if follower.user.id not in user_follow_dates.keys():
                user_follow_dates[follower.user.id] = []
            user_follow_dates[follower.user.id].append(follower.followed_at)

In [ ]:
deltas = []
for (user_id, follows) in user_follow_dates.items():
    if len(follows) > 1:
        sf = sorted(follows)
        deltas.append((sf[1] - sf[0]))

In [ ]:
len(deltas), np.average(deltas), np.median(deltas), np.sum(np.array(deltas) < timedelta(days=180)) / len(deltas), np.sum(np.array(deltas) < timedelta(days=360)) / len(deltas)

In [ ]:
df = pd.DataFrame({
    "x": filter(
        lambda el: el < 100,
        list(map(lambda el: el.days, deltas))
    )
})
fig = px.histogram(df, 
                   title="Через сколько дней происходит вторая подписка на чубу",
                   x="x")
fig.show() 

In [ ]:
df = pd.DataFrame({
    "x": list(map(lambda user: len(data.user_following[user]), data.user_following.keys())),
})
fig = px.histogram(df,
                   title="Сколько витуберов смотрят зрители", 
                   x="x")
fig.show() 

### Уникальность витуберов

Посмотрим, какая часть аудитории витубера смотрит только его 

In [ ]:
len(data.user_following), sum(map(lambda el: len(el[1]), data.user_following.items()))

In [ ]:
follows_counts = list(map(lambda fd: len(fd[1]), data.user_following.items()))
follows_counts = np.array(follows_counts)
len(follows_counts), np.average(follows_counts), np.median(follows_counts)

In [ ]:
for i in range(1, 5):
    print(f"{i})", np.sum(follows_counts == i) / len(follows_counts) * 100, "%")
print(f"{12}+)", np.sum(follows_counts > 11) / len(follows_counts) * 100, "%", np.sum(follows_counts > 11))


In [ ]:
unique_users = list(filter(lambda fd: len(fd[1]) == 1, data.user_following.items()))
print(len(unique_users) / len(data.user_following) * 100, "%")

In [ ]:
unique_users2 = list(filter(lambda fd: len(fd[1]) == 2, data.user_following.items()))
print(len(unique_users2) / len(data.user_following) * 100, "%")

In [ ]:
unique_users3 = list(filter(lambda fd: len(fd[1]) == 3, data.user_following.items()))
print(len(unique_users3) / len(data.user_following) * 100, "%")

In [ ]:
unique_users3 = list(filter(lambda fd: len(fd[1]) > 3, data.user_following.items()))
print(len(unique_users3) / len(data.user_following) * 100, "%")

In [ ]:
RANGE_MIN = 1
RANGE_MAX = 2
unique_users = list(filter(lambda fd: len(fd[1]) in range(RANGE_MIN, RANGE_MAX), data.user_following.items()))
vtuber_unique_followers_counter = {}
for (user, following) in unique_users:
    for vtuber_name in following:
        vtuber_unique_followers_counter[vtuber_name] = vtuber_unique_followers_counter.get(vtuber_name, 0) + 1

vtuber_unique_followers_proportion = {}
for vtuber in data.vtuber_names:
    vtuber_unique_followers_proportion[vtuber] = 0
for (vtuber, unique_followers) in vtuber_unique_followers_counter.items():
    vtuber_unique_followers_proportion[vtuber] = unique_followers / len(data.vtuber_followers[vtuber]) * 100

In [ ]:
followers_count_data = np.array(list(map(lambda vtuber: len(data.vtuber_followers[vtuber]), data.vtuber_names)))
unique_followers_proportion_data = np.array(list(map(lambda vtuber: vtuber_unique_followers_proportion[vtuber], data.vtuber_names)))
df = pd.DataFrame({
    "Followers": followers_count_data,
    "Unique followers (%)": unique_followers_proportion_data,
    "vtuber": data.vtuber_names,
})
fig = px.scatter(df, x="Followers", y="Unique followers (%)", hover_name="vtuber")
# fig.write_image(f"vtubers_uniqueness_{RANGE_MIN}_{RANGE_MAX}.png", width=1980, height=1080)
# fig.write_html(f"vtubers_uniqueness_{RANGE_MIN}_{RANGE_MAX}.html")
fig.show()

In [ ]:
len(unique_users), np.median(unique_followers_proportion_data), np.percentile(unique_followers_proportion_data, 60), np.percentile(unique_followers_proportion_data, 90)

In [ ]:
fig = px.histogram(df, x="Unique followers (%)", nbins=100)
# fig.write_image(f"vtubers_uniqueness_{RANGE_MIN}_{RANGE_MAX}_histogram.png", width=1980, height=1080)
fig.show()

In [ ]:
fig = px.histogram(df, x="Unique followers (%)", nbins=100, cumulative=True, histnorm="percent")
# fig.write_image(f"vtubers_uniqueness_{RANGE_MIN}_{RANGE_MAX}_histogram_cumulative.png", width=1980, height=1080)
fig.show()

In [ ]:
x_data = []
y_data = []

for i in range(12, 1, -1):
    RANGE_MIN = i
    RANGE_MAX = 2000
    unique_users = list(filter(lambda fd: len(fd[1]) in range(RANGE_MIN, RANGE_MAX), data.user_following.items()))
    x_data.append(i)
    y_data.append(len(unique_users))
df = pd.DataFrame({
    "N": x_data,
    "Core Auditory Size": y_data,
})

fig = px.line(df, x="N", y="Core Auditory Size", range_y=(0, 310_000), title="Core Auditory Size")
# fig.write_image(f"core_auditory_size.png", width=600, height=400)
fig.show()

In [ ]:
np.percentile(unique_followers_proportion_data, 75), \
    np.percentile(unique_followers_proportion_data, 80), \
    np.percentile(unique_followers_proportion_data, 95)

In [ ]:
data.vtuber_names = np.array(data.vtuber_names)

Уникальность разных ютуберов:

In [ ]:
print('megubite', followers_count_data[data.vtuber_names == 'megubite'], unique_followers_proportion_data[data.vtuber_names == 'megubite'])
print('podvalmishki', followers_count_data[data.vtuber_names == 'podvalmishki'], unique_followers_proportion_data[data.vtuber_names == 'podvalmishki'])
print('nurich0', followers_count_data[data.vtuber_names == 'nurich0'], unique_followers_proportion_data[data.vtuber_names == 'nurich0'])
print('eboni_inobe', followers_count_data[data.vtuber_names == 'eboni_inobe'], unique_followers_proportion_data[data.vtuber_names == 'eboni_inobe'])
print('marts_comm', followers_count_data[data.vtuber_names == 'marts_comm'], unique_followers_proportion_data[data.vtuber_names == 'marts_comm'])
print('novikony_', followers_count_data[data.vtuber_names == 'novikony_'], unique_followers_proportion_data[data.vtuber_names == 'novikony_'])
print('crabcyborg', followers_count_data[data.vtuber_names == 'crabcyborg'], unique_followers_proportion_data[data.vtuber_names == 'crabcyborg'])
print('manarenewal', followers_count_data[data.vtuber_names == 'manarenewal'], unique_followers_proportion_data[data.vtuber_names == 'manarenewal'])
print('snezha_mrr', followers_count_data[data.vtuber_names == 'snezha_mrr'], unique_followers_proportion_data[data.vtuber_names == 'snezha_mrr'])
print('mariodiamato', followers_count_data[data.vtuber_names == 'mariodiamato'], unique_followers_proportion_data[data.vtuber_names == 'mariodiamato'])
print('kashtan_mp4', followers_count_data[data.vtuber_names == 'kashtan_mp4'], unique_followers_proportion_data[data.vtuber_names == 'kashtan_mp4'])
print('ryrver', followers_count_data[data.vtuber_names == 'ryrver'], unique_followers_proportion_data[data.vtuber_names == 'ryrver'])
print('erumi333', followers_count_data[data.vtuber_names == 'erumi333'], unique_followers_proportion_data[data.vtuber_names == 'erumi333'])
print('tarelko', followers_count_data[data.vtuber_names == 'tarelko'], unique_followers_proportion_data[data.vtuber_names == 'tarelko'])
print('yodice_', followers_count_data[data.vtuber_names == 'yodice_'], unique_followers_proportion_data[data.vtuber_names == 'yodice_'])
print('the_lerachi', followers_count_data[data.vtuber_names == 'the_lerachi'], unique_followers_proportion_data[data.vtuber_names == 'the_lerachi'])

Уникальность разых парней витуберов

In [ ]:
print('aerochannelekat', followers_count_data[data.vtuber_names == 'aerochannelekat'], unique_followers_proportion_data[data.vtuber_names == 'aerochannelekat'])
print('kittenblaze', followers_count_data[data.vtuber_names == 'kittenblaze'], unique_followers_proportion_data[data.vtuber_names == 'kittenblaze'])
print('maxsych_', followers_count_data[data.vtuber_names == 'maxsych_'], unique_followers_proportion_data[data.vtuber_names == 'maxsych_'])
print('mariodiamato', followers_count_data[data.vtuber_names == 'mariodiamato'], unique_followers_proportion_data[data.vtuber_names == 'mariodiamato'])
print('igor_vich', followers_count_data[data.vtuber_names == 'igor_vich'], unique_followers_proportion_data[data.vtuber_names == 'igor_vich'])
print('ryrver', followers_count_data[data.vtuber_names == 'ryrver'], unique_followers_proportion_data[data.vtuber_names == 'ryrver'])
print('ravenyan', followers_count_data[data.vtuber_names == 'ravenyan'], unique_followers_proportion_data[data.vtuber_names == 'ravenyan'])
print('keensh1r0', followers_count_data[data.vtuber_names == 'keensh1r0'], unique_followers_proportion_data[data.vtuber_names == 'keensh1r0'])
print('fibi_ch', followers_count_data[data.vtuber_names == 'fibi_ch'], unique_followers_proportion_data[data.vtuber_names == 'fibi_ch'])
print('kibcharnnel', followers_count_data[data.vtuber_names == 'kibcharnnel'], unique_followers_proportion_data[data.vtuber_names == 'kibcharnnel'])
print('farostg', followers_count_data[data.vtuber_names == 'farostg'], unique_followers_proportion_data[data.vtuber_names == 'farostg'])
print('egorzerotwo', followers_count_data[data.vtuber_names == 'egorzerotwo'], unique_followers_proportion_data[data.vtuber_names == 'egorzerotwo'])
print('ezdedus', followers_count_data[data.vtuber_names == 'ezdedus'], unique_followers_proportion_data[data.vtuber_names == 'ezdedus'])
print('nyapuru', followers_count_data[data.vtuber_names == 'nyapuru'], unique_followers_proportion_data[data.vtuber_names == 'nyapuru'])
print('cirnostrong', followers_count_data[data.vtuber_names == 'cirnostrong'], unique_followers_proportion_data[data.vtuber_names == 'cirnostrong'])
print('raw_rawfler', followers_count_data[data.vtuber_names == 'raw_rawfler'], unique_followers_proportion_data[data.vtuber_names == 'raw_rawfler'])
print('ixxxiq', followers_count_data[data.vtuber_names == 'ixxxiq'], unique_followers_proportion_data[data.vtuber_names == 'ixxxiq'])
print('ericmyval', followers_count_data[data.vtuber_names == 'ericmyval'], unique_followers_proportion_data[data.vtuber_names == 'ericmyval'])

### Кластеризация витуберов

#### Векторизация данных

In [ ]:
columns = data.user_following.keys()
user_id_to_column = {}
for (i, col) in zip(range(len(columns)), columns):
    user_id_to_column[col] = i

def vtuber2vec(vtuber: str) -> np.ndarray:
    result = np.zeros(len(data.user_following))
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None:
            result[user_id_to_column[follower.user.id]] = 1
    
    return result

In [ ]:
dataset = []
for vtuber in data.vtuber_names:
    dataset.append(vtuber2vec(vtuber))
dataset = np.array(dataset)

#### Кластеризация и отрисвка

##### Косинусное расстояние

In [ ]:
projection = sklearn.manifold.TSNE(metric="cosine", verbose=2, random_state=1).fit_transform(dataset)

In [ ]:
labels = sklearn.cluster.HDBSCAN(min_cluster_size=2, max_cluster_size=100, metric="cosine").fit_predict(X=dataset)
# eps = 0.92
# labels = sklearn.cluster.DBSCAN(eps=eps, min_samples=2, metric="cosine").fit_predict(X=dataset)

In [ ]:
data_frame = pd.DataFrame({
    "x": projection[:, 0],
    "y": projection[:, 1],
    "Кластер": labels,
    "Имя": data.vtuber_names
})
data_frame["Кластер"] = data_frame["Кластер"].astype(str) #convert to string

In [ ]:
fig = px.scatter(data_frame, 
                 x="x", 
                 y="y",
                 color="Кластер", 
                 hover_name="Имя")
# fig.write_html(f"vtuber_clusterisation_DBSCAN_{eps}.html")
# fig.write_html(f"vtuber_clusterisation_HDBSCAN_2+_white.png", width=1280, height=720)
fig.show()

Вывести всех не из нулевого кластера(нужно для поиска уникалов)

In [ ]:
np.array(data.vtuber_names)[labels != 0]

Уникальность с подсвеченными кникалами (те, кто не из нулевого кластера)

In [ ]:
RANGE_MIN = 1
RANGE_MAX = 2
unique_users = list(filter(lambda fd: len(fd[1]) in range(RANGE_MIN, RANGE_MAX), data.user_following.items()))
vtuber_unique_followers_counter = {}
for (user, following) in unique_users:
    for vtuber_name in following:
        vtuber_unique_followers_counter[vtuber_name] = vtuber_unique_followers_counter.get(vtuber_name, 0) + 1

vtuber_unique_followers_proportion = {}
for vtuber in data.vtuber_names:
    vtuber_unique_followers_proportion[vtuber] = 0
for (vtuber, unique_followers) in vtuber_unique_followers_counter.items():
    vtuber_unique_followers_proportion[vtuber] = unique_followers / len(data.vtuber_followers[vtuber]) * 100

followers_count_data = np.array(list(map(lambda vtuber: len(data.vtuber_followers[vtuber]), data.vtuber_names)))
unique_followers_proportion_data = np.array(list(map(lambda vtuber: vtuber_unique_followers_proportion[vtuber], data.vtuber_names)))
df = pd.DataFrame({
    "Followers": followers_count_data,
    "Unique followers (%)": unique_followers_proportion_data,
    "vtuber": data.vtuber_names,
    "color": (labels != 0),
})
fig = px.scatter(df, x="Followers", y="Unique followers (%)", hover_name="vtuber", color="color")
# fig.write_html(f"vtuber_uniqueness_with_clustering_uniqueness.html")
fig.show()

Вывод всех кластеров размера > 2

In [ ]:
for i in range(max(labels) + 1):
    if len(np.array(data.vtuber_names)[labels == i]) > 2:
        print(i, np.array(data.vtuber_names)[labels == i])

In [ ]:
def cosine_metric(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return 1 - np.dot(vec1, vec2) / np.sqrt(np.sum(vec1**2)) / np.sqrt(np.sum(vec2**2))

def manhattan_metric(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return np.sum(np.abs(vec1 - vec2))

def distance(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return np.sum(np.abs(vec1 * vec2))

def jaccard(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return np.sum(np.abs(vec1 * vec2)) / np.sum(np.logical_or(vec1, vec2).astype(int))

def simple_metric(vec1: np.ndarray, vec2: np.ndarray) -> float:
    inter = np.sum(vec1 * vec2)
    a = np.sum(vec1)
    b = np.sum(vec2)
    return 1 - inter / (a + b - inter)

def podsos(vec1: np.ndarray, vec2: np.ndarray) -> float:
    inter = np.sum(vec1 * vec2)
    a = np.sum(vec1)
    b = np.sum(vec2)
    return 1 - inter / (min(a, b))

def podsos_asymentric(vec1: np.ndarray, vec2: np.ndarray) -> float:
    inter = np.sum(vec1 * vec2)
    a = np.sum(vec1)
    b = np.sum(vec2)
    return 1 - inter / a

def intersection_absolute(vec1: np.ndarray, vec2: np.ndarray) -> float:
    inter = np.sum(vec1 * vec2)
    return inter

In [ ]:
METRIC = intersection_absolute
j = data.vtuber_names.index('fra3a')
result = []

for i in range(len(data.vtuber_names)):
    result.append((data.vtuber_names[i], METRIC(dataset[i], dataset[j])))

In [ ]:
from pprint import pprint

pprint(list(sorted(result, key=lambda el: el[1])))

In [ ]:
MATRIX = np.zeros((len(data.vtuber_names), len(data.vtuber_names)))
METRIC = podsos_asymentric

for i in range(len(data.vtuber_names)):
    print(f"{i})", data.vtuber_names[i])

    for j in range(len(data.vtuber_names)):
        vtuber1_vec = dataset[i]
        vtuber2_vec = dataset[j]
        distance = METRIC(vtuber1_vec, vtuber2_vec)
        MATRIX[i][j] = distance

In [ ]:
# MAX_DISTANCE = 0.78
MAX_DISTANCE = 0.50
EDGES = []

for i in range(len(data.vtuber_names)):
    for j in range(i + 1, len(data.vtuber_names)):
        if (MATRIX[i][j] < MAX_DISTANCE or MATRIX[j][i] < MAX_DISTANCE) \
                and data.vtuber_names[i] not in IGNORED_VTUBERS \
                and data.vtuber_names[j] not in IGNORED_VTUBERS:
            EDGES.append((i, j))
print("MAX_DISTANCE:", MAX_DISTANCE)
print("Edges:", len(EDGES))

fig = go.Figure()

for (i, j) in EDGES:
    projection1 = projection[i]
    projection2 = projection[j]
    fig.add_trace(go.Scatter(x=[projection1[0], projection2[0]],
                            y=[projection1[1], projection2[1]],
                            mode="lines",
                            marker_color="#ffffff"))
colorscale = ['aggrnyl', 'agsunset', 'algae', 'amp', 'armyrose', 'balance',
    'blackbody', 'bluered', 'blues', 'blugrn', 'bluyl', 'brbg', 'brwnyl', 
    'bugn', 'bupu', 'burg', 'burgyl', 'cividis', 'curl', 'darkmint', 'deep', 
    'delta', 'dense', 'earth', 'edge', 'electric', 'emrld', 'fall', 'geyser', 
    'gnbu', 'gray', 'greens', 'greys', 'haline', 'hot', 'hsv', 'ice', 'icefire', 
    'inferno', 'jet', 'magenta', 'magma', 'matter', 'mint', 'mrybm', 'mygbm', 
    'oranges', 'orrd', 'oryel', 'oxy', 'peach', 'phase', 'picnic', 'pinkyl', 
    'piyg', 'plasma', 'plotly3', 'portland', 'prgn', 'pubu', 'pubugn', 'puor', 
    'purd', 'purp', 'purples', 'purpor', 'rainbow', 'rdbu', 'rdgy', 'rdpu', 
    'rdylbu', 'rdylgn', 'redor', 'reds', 'solar', 'spectral', 'speed', 'sunset', 
    'sunsetdark', 'teal', 'tealgrn', 'tealrose', 'tempo', 'temps', 'thermal', 
    'tropic', 'turbid', 'turbo', 'twilight', 'viridis', 'ylgn', 'ylgnbu', 
    'ylorbr', 'ylorrd'][1] # [39]
fig.add_trace(go.Scatter(x=projection[:, 0], 
                        y=projection[:, 1],
                        marker=dict(
                            color=labels,
                            colorscale=colorscale,
                        ),
                        text=data.vtuber_names,
                        mode="markers"))
    
# fig.write_image(f"podsos_edges_{MAX_DISTANCE}.png", width=1980, height=1080)
# fig.write_html(f"podsos_edges_{MAX_DISTANCE}.html")
fig.show()

In [ ]:
vtubers_intersections = []
vtubers_intersections_names = []
for i in range(len(data.vtuber_names)):
    for j in range(i + 1, len(data.vtuber_names)):
        if data.vtuber_names[i] not in IGNORED_VTUBERS and data.vtuber_names[j] not in IGNORED_VTUBERS:
            vtubers_intersections.append(1 - min(MATRIX[i][j], MATRIX[j][i]))
            vtubers_intersections_names.append((data.vtuber_names[i], data.vtuber_names[j]))
vtubers_intersections = np.array(vtubers_intersections)
vtubers_intersections_names = np.array(vtubers_intersections_names)

df = pd.DataFrame({
    "Пересечение (%)": vtubers_intersections * 100,
})
fig = px.histogram(df, x="Пересечение (%)", title="Распределение пересечений в рувитубе", nbins=100, range_x=(0, 100))
# fig.write_image(f"intersections_all.png", width=1980, height=1080)
# fig.write_html(f"intersections_all.html")
fig.show() 

In [ ]:
from pprint import pprint
limit = 0.5
pprint(list(sorted(np.column_stack((vtubers_intersections[vtubers_intersections > limit], vtubers_intersections_names[vtubers_intersections > limit])), key=lambda el: -float(el[0]))))

### Анализ количества фолловов у зрителей

#### Количество глобальных фолловов

In [14]:
unique_users = []
used_logins = set()
for vtuber in data.vtuber_names:
    for follower_data in data.vtuber_followers[vtuber]:
        if follower_data.user is not None and follower_data.user.login not in used_logins:
            used_logins.add(follower_data.user.login)
            unique_users.append(follower_data.user)

In [15]:
draw_data = unique_users
draw_data = list(map(lambda el: el.follows_count, draw_data))
draw_data = list(filter(lambda el: el < 500, draw_data))
draw_data = np.array(draw_data)
df = pd.DataFrame({
    "Following": draw_data
})
print(len(draw_data), np.average(draw_data), np.median(draw_data))
fig = px.histogram(df, x="Following")
fig.show()

1412710 58.72155785688499 24.0


#### Количество локальных фолловов

In [16]:
draw_data = data.user_following.items()
draw_data = list(map(lambda el: el[1], draw_data))
draw_data = list(map(lambda el: len(el), draw_data))
draw_data = list(filter(lambda el: el > 0, draw_data))
df = pd.DataFrame({
    "Following": draw_data
})
fig = px.histogram(df, x="Following")
fig.show()

#### Количество абсолютных уникалов среди фолловеров

In [17]:
follows_count = []
abs_unique = []
for vtuber in data.vtuber_names:
    follows_count.append(len(data.vtuber_followers[vtuber]))
for vtuber in data.vtuber_names:
    vtuber_abs_unique = 0
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None and follower.user.follows_count == 1:
            vtuber_abs_unique += 1
    abs_unique.append(vtuber_abs_unique)
abs_unique = np.array(abs_unique)
follows_count = np.array(follows_count)

df = pd.DataFrame({
    "Absolutely unique followers (%)": abs_unique / follows_count * 100,
    "Followers": follows_count,
    "Vtuber": vtuber_names,
})
fig = px.scatter(df, x="Followers", y="Absolutely unique followers (%)", hover_name="Vtuber")
fig.show()

In [18]:
df = pd.DataFrame({
    "Absolutely unique followers": abs_unique,
    "Followers": follows_count,
    "Vtuber": vtuber_names,
})
fig = px.scatter(df, x="Followers", y="Absolutely unique followers", hover_name="Vtuber")
fig.show()

## Аналитика сообщений

In [ ]:
@dataclass
class EmoteData:
    id: str


@dataclass
class MessageFragmentData:
    emote: Optional[EmoteData]
    text: Optional[str]
    mention: Optional[UserData]

    def as_text(self) -> str:
        if self.emote is not None:
            return f"<@{self.emote.id}>"
        elif self.text is not None:
            return self.text
        elif self.mention is not None:
            return f"@{self.mention.login}"
        else:
            return ""

    def as_plain_text(self) -> str:
        if self.emote is not None:
            return ""
        elif self.text is not None:
            return self.text
        elif self.mention is not None:
            return ""
        else:
            return ""


@dataclass
class MessageData:
    as_text: str
    as_plain_text: str
    fragments: List[MessageFragmentData]


@dataclass
class CommentData:
    id: str
    commenter: UserData
    contentOffsetSeconds: int
    message: MessageData


@dataclass
class VideoData:
    id: int
    title: str
    description: str
    created_at: datetime
    view_count: int
    comments: List[CommentData]


@dataclass
class VtuberVideosData:
    vtuber_names: List[str]
    vtuber_videos: Dict[str, List[VideoData]]

In [ ]:
def json_to_fragment_data(json_data) -> MessageFragmentData:
    return MessageFragmentData(emote=None if json_data['emote'] is None else EmoteData(id=json_data['emote']['id']),
                               text=json_data['text'],
                               mention=None if json_data['mention'] is None else json_to_user_data(json_data['mention']))


def fragment_data_to_json(fragment_data: MessageFragmentData):
    return {
        'emote': None if fragment_data.emote is None else {
            'id': fragment_data.emote.id
        },
        'text': fragment_data.text,
        'mention': None if fragment_data.mention is None else user_data_to_json(fragment_data.mention)
    }


def json_to_message_data(json_data) -> MessageData:
    as_text = ""
    as_plain_text = ""
    fragments = []

    for fragment_json in json_data['fragments']:
        fragment = json_to_fragment_data(fragment_json)
        as_text += fragment.as_text()
        as_plain_text += fragment.as_plain_text()
        fragments.append(fragment)

    return MessageData(as_text=as_text,
                       as_plain_text=as_plain_text,
                       fragments=fragments)


def message_data_to_json(message_data: MessageData):
    return {
        'fragments': list(map(fragment_data_to_json, message_data.fragments))
    }


def json_to_comment_data(json_data) -> CommentData:
    return CommentData(id=json_data['id'],
                       commenter=None if json_data['commenter'] is None else json_to_user_data(json_data['commenter']),
                       contentOffsetSeconds=int(json_data['contentOffsetSeconds']),
                       message=json_to_message_data(json_data['message']))


def user_data_to_json(user_data: UserData):
    return {
        'id': user_data.id,
        'login': user_data.login,
        'createdAt': None if user_data.created_at is None else str(user_data.created_at),
        'deletedAt': None if user_data.deleted_at is None else str(user_data.deleted_at),
        'follows': {
            'totalCount': str(user_data.follows_count)
        }
    }


def comment_data_to_json(comment_data: CommentData):
    return {
        'id': comment_data.id,
        'commenter': None if comment_data.commenter is None else user_data_to_json(comment_data.commenter),
        'contentOffsetSeconds': str(comment_data.contentOffsetSeconds),
        'message': message_data_to_json(comment_data.message)
    }


def json_to_video_data(json_data) -> VideoData:
    comments = []
    for comment in json_data['comments']['edges']:
        comments.append(json_to_comment_data(comment['node']))

    return VideoData(id=int(json_data['id']),
                     title=json_data['title'],
                     description=json_data['description'],
                     created_at=None if json_data['createdAt'] is None else datetime.fromisoformat(json_data['createdAt']),
                     view_count=int(json_data['viewCount']),
                     comments=comments)


def video_data_to_json(video_data: VideoData):
    return {
        'id': str(video_data.id),
        'title': video_data.title,
        'description': video_data.description,
        'createdAt': None if video_data.created_at is None else str(video_data.created_at),
        'viewCount': str(video_data.view_count),
        'comments': {
            'edges': list(map(
                lambda comment_data: {
                    'node': comment_data_to_json(comment_data)
                }, 
                video_data.comments))
        },
    }


def join_video_data(left: Optional[VideoData], right: VideoData) -> VideoData:
    if left is None:
        return right
    else:
        assert left.id == right.id

        comments = []
        comments.extend(left.comments)
        comments.extend(right.comments)

        return VideoData(id=left.id,
                         title=left.title,
                         description=left.description,
                         created_at=left.created_at,
                         view_count=left.view_count,
                         comments=comments)

In [ ]:
import requests
import time
import json
import pprint


API_URL = "https://gql.twitch.tv/gql"
API_CLIENT_ID = "kd1unb4b3q4t58fwlpcbzcbnm76a8fp"
API_REQUEST_DATA = """
query fetchVideoData($id: ID, $first: Int, $after: Cursor) {
    video(id: $id) {
        id
        title
        description
        createdAt
        viewCount
        comments(first: $first, after: $after) {
            edges {
                cursor
                node {
                    id
                    commenter {
                        id
                        login
                        createdAt
                        deletedAt
                        follows {
                            totalCount
                        }
                    }
                    contentOffsetSeconds
                    message {
                        fragments {
                            emote {
                                id
                            }
                            mention {
                                id
                                login
                                createdAt
                                deletedAt
                                follows {
                                    totalCount
                                }
                            }
                            text
                        }
                    }
                }
            }
            pageInfo {
                hasNextPage
            }
        }
    }
}
"""


def send_request(session: requests.Session, 
                 id: int, 
                 cursor: str,
                 log: bool,
                 repeat_times: int,
                 repeat_delay: float):
    for i in range(repeat_times):
        if i > 0:
            time.sleep(repeat_delay)

            if log:
                print("[LOG]", "Repeat last request")

        response = session.post(
            url=API_URL,
            json={
                'query': API_REQUEST_DATA,
                'variables': {
                    'id': str(id),
                    'first': 100,
                    'after': cursor
                }
            },
            headers={
                "Client-ID": API_CLIENT_ID
            })
        
        if response.status_code != 200:
            if log:
                print("[ERROR]", "Status code =", response.status_code)
            continue

        data = json.loads(response.text)
        if 'errors' in data.keys() and len(data['errors']) > 0:
            for error in data['errors']:
                if log:
                    print("[ERROR]", error)
            continue

        return response
    
    return None


def load_video_data(id: int, log: bool = False, repeat_times: int = 5, repeat_delay: float = 1.0) -> Optional[VideoData]:
    cache_path = f"./data/.temp/video/{id}.json"

    if os.path.exists(cache_path):
        if log:
            print("[LOG]", "Loading from the cache")
        with open(cache_path, 'r') as cache_file:
            return json_to_video_data(json.load(cache_file))

    session = requests.Session()
    cursor = None
    result = None

    while True:
        response = send_request(session, id, cursor, log, repeat_times, repeat_delay)

        if response is None:
            if log:
                print("[ERROR]", "Failed load", id)
            return None

        response_data = json.loads(response.text)

        if response_data['data']['video'] is None:
            if log:
                print("[ERROR]", "Failed load", id)
            return None
        
        result = join_video_data(result, json_to_video_data(response_data['data']['video']))
        

        if log:
            if len(result.comments) > 0:
                print(f"[LOG|{id}]", "Loaded", 
                      result.comments[-1].contentOffsetSeconds // 60 // 60, "hours", 
                      result.comments[-1].contentOffsetSeconds // 60 % 60, "minutes", 
                      result.comments[-1].contentOffsetSeconds % 60, "seconds")
            else:
                print(f"[LOG|{id}] Loaded ...")

        if len(response_data['data']['video']['comments']['edges']) == 0:
            break
        else:
            cursor = response_data['data']['video']['comments']['edges'][-1]['cursor']

        if not response_data['data']['video']['comments']['pageInfo']['hasNextPage'] or cursor == "":
            break

    if log:
        print('[LOG]', "caching")
    with open(cache_path, 'w') as cache_file:
        json.dump(video_data_to_json(result), cache_file, indent=4)

    return result
        


In [ ]:
API_REQUEST_DATA_2 = """
query fetchVideoData($login: String, $first: Int, $after: Cursor) {
    user(login: $login) {
        videos(first: $first, after: $after, type: ARCHIVE, sort: TIME) {
            totalCount
            pageInfo {
                hasNextPage
            }
            edges {
                cursor
                node {
                    id
                }
            }
        }
    }
}
"""


def send_request_2(session: requests.Session, 
                   login: str, 
                   cursor: str,
                   log: bool,
                   repeat_times: int,
                   repeat_delay: float):
    for i in range(repeat_times):
        if i > 0:
            time.sleep(repeat_delay)

            if log:
                print("[LOG]", "Repeat last request")

        response = session.post(
            url=API_URL,
            json={
                'query': API_REQUEST_DATA_2,
                'variables': {
                    'login': login,
                    'first': 20,
                    'after': cursor
                }
            },
            headers={
                "Client-ID": API_CLIENT_ID
            })
        
        if response.status_code != 200:
            if log:
                print("[ERROR]", "Status code =", response.status_code)
            continue

        data = json.loads(response.text)
        if 'errors' in data.keys() and len(data['errors']) > 0:
            for error in data['errors']:
                if log:
                    print("[ERROR]", error)
            continue

        return response
    
    return None


def load_streamer_videos(login: str, log: bool = False, repeat_times: int = 5, repeat_delay: float = 1.0, cacheonly: bool = False) -> Optional[List[int]]:
    result = []
    cache_path = f"./data/.temp/videos/{login}.json"

    if os.path.exists(cache_path):
        if log:
            print("[LOG]", "Loading from the cache")
        with open(cache_path, 'r') as cache_file:
            result = json.load(cache_file)

    if cacheonly:
        return result

    session = requests.Session()
    cursor = None

    while True:
        response = send_request_2(session, login, cursor, log, repeat_times, repeat_delay)
        assert response is not None

        response_data = json.loads(response.text)

        if response_data['data']['user']['videos'] is None:
            if log:
                print("[ERROR] Ivalid streamer:", login)
            break

        if response_data['data']['user']['videos']['edges'] is not None:
            for video_data in response_data['data']['user']['videos']['edges']:
                cursor = video_data['cursor']
                id = int(video_data['node']['id'])

                if id not in result:
                    result.append(id)
        
        if not response_data['data']['user']['videos']['pageInfo']['hasNextPage'] or cursor == "":
            break

    if log:
        print("[LOG]", "Caching result")
    with open(cache_path, 'w') as cache_file:
        json.dump(result, cache_file, indent=4)

    return result

In [ ]:
def load_vtuber_video(vtuber_login: str, video_id: int) -> Tuple[str, Optional[VideoData]]:
    assert load_video_data(video_id, log=True, repeat_times=10, repeat_delay=5.0) is not None


vtuber_videos = {}
with ThreadPoolExecutor(max_workers=64) as executor:
    futures = []

    for vtuber in vtuber_names:
        streamer_videos = load_streamer_videos(vtuber, log=True)
        assert streamer_videos is not None
        for video_id in streamer_videos:
            future = executor.submit(load_vtuber_video, vtuber_login=vtuber, video_id=video_id)
            futures.append(future)
    
    for future in as_completed(futures):
        future.result()

In [ ]:
# @dataclass
# class VtuberVideosData:
#     vtuber_names: List[str]
#     vtuber_videos: Dict[str, List[VideoData]]

vtuber_videos = {}
for vtuber in vtuber_names:
    vtuber_videos[vtuber] = []

    for video_id in load_streamer_videos(vtuber, log=True, cacheonly=True):
        video_data = load_video_data(video_id, log=True)
        vtuber_videos[vtuber].append(video_data)

In [ ]:
vvdata = VtuberVideosData(vtuber_names=vtuber_names,
                          vtuber_videos=vtuber_videos)

In [ ]:
messages_amount = 0
for vtuber in vvdata.vtuber_names:
    for video_data in vvdata.vtuber_videos[vtuber]:
        messages_amount += len(video_data.comments)
messages_amount

In [ ]:
commenters = set()
for vtuber in vvdata.vtuber_names:
    for video_data in vvdata.vtuber_videos[vtuber]:
        for comment in video_data.comments:
            if comment.commenter is not None:
                commenters.add(comment.commenter.id)
len(commenters)

In [ ]:
commenter_counters = {}
commenter_logins = {}
for commenter in commenters:
    commenter_counters[commenter] = 0
for vtuber in vvdata.vtuber_names:
    for video_data in vvdata.vtuber_videos[vtuber]:
        for comment in video_data.comments:
            if comment.commenter is not None:
                commenter_counters[comment.commenter.id] += 1
                commenter_logins[comment.commenter.id] = comment.commenter.login

In [ ]:
import random

FILTER_IDS = [100135110, 19264788, 1564983]


random.seed(6)
fig = go.Figure()


def not_ignore_comment_data(comment_data: CommentData) -> bool:
    return comment_data.commenter is None or comment_data.commenter.id not in FILTER_IDS


def add_plot(fig, streamer):
    max_index = len(vvdata.vtuber_videos[streamer])
    stream_data = vvdata.vtuber_videos[streamer][random.randint(0, max_index - 1)]
    print(streamer, ":", stream_data.id, stream_data.title, stream_data.created_at)
    
    plot_data = stream_data.comments
    plot_data = list(filter(not_ignore_comment_data, plot_data))
    plot_data = list(map(lambda comment: comment.contentOffsetSeconds // 600, plot_data))
    minutes = list(range(max(plot_data) + 1))
    counts = [0] * (max(plot_data) + 1)
    for el in plot_data:
        counts[el] += 1
    minutes = np.array(minutes)
    counts = np.array(counts)
    fig.add_trace(go.Scatter(x=minutes, 
                             y=counts,
                             mode='lines+markers',
                             name=streamer))

add_plot(fig, "xkamysh")
add_plot(fig, "qchaan_9")
add_plot(fig, "rera_seal")
add_plot(fig, "nyamuras")
add_plot(fig, "alaskanyan")
add_plot(fig, "yui2d")
add_plot(fig, "trixie_vox")
add_plot(fig, "akane_iwawwa")

fig.show()

In [ ]:
import random

random.seed(6)
fig = go.Figure()

def add_plot(fig, streamer):
    weights = []

    for stream_data in vvdata.vtuber_videos[streamer]:
        max_second = max(map(lambda comment: comment.contentOffsetSeconds, stream_data.comments))
        while len(weights) - 1 < max_second:
            weights.append(0)
        for i in range(max_second + 1):
            weights[i] += 1

    comments_counts = [0] * len(weights)
    for stream_data in vvdata.vtuber_videos[streamer]:
        for comment  in stream_data.comments:
            if not_ignore_comment_data(comment):
                comments_counts[comment.contentOffsetSeconds] += 1
    
    weights = np.array(weights)
    comments_counts = np.array(comments_counts)
    weighed_comments_counts = comments_counts / weights
    group_size = 300
    grouped_weighed_comments_counts = []
    for i in range(len(weighed_comments_counts)):
        if len(grouped_weighed_comments_counts) <= i // group_size:
            grouped_weighed_comments_counts.append(0)
        grouped_weighed_comments_counts[i // group_size] += weighed_comments_counts[i]
    times = np.array(range(len(weights)))

    fig.add_trace(go.Scatter(x=times, 
                             y=grouped_weighed_comments_counts,
                             mode='lines+markers',
                             name=streamer))

add_plot(fig, "xkamysh")
add_plot(fig, "qchaan_9")
add_plot(fig, "rera_seal")
add_plot(fig, "nyamuras")
add_plot(fig, "alaskanyan")
add_plot(fig, "yui2d")
add_plot(fig, "trixie_vox")
add_plot(fig, "akane_iwawwa")
add_plot(fig, "nubchann")
add_plot(fig, "planyach")
add_plot(fig, "banshameow")
add_plot(fig, "itsmoriko")
add_plot(fig, "severinasoda")
add_plot(fig, "snezha_mrr")
add_plot(fig, "conway671")
add_plot(fig, "ezdedus")

fig.show()


In [ ]:
GROUP_SIZE = 300 # 5 minutes
MAX_GROUP_COUNT = 36 # 3 hours


def get_streamer_chat_data(streamer: str) -> np.ndarray:
    weights = np.zeros(MAX_GROUP_COUNT * GROUP_SIZE)

    for stream_data in vvdata.vtuber_videos[streamer]:
        if len(stream_data.comments) == 0:
            max_second = 0
        else:
            max_second = max(map(lambda comment: comment.contentOffsetSeconds, stream_data.comments))
        for i in range(min(max_second + 1, MAX_GROUP_COUNT * GROUP_SIZE)):
            weights[i] += 1

    comments_counts = np.zeros(MAX_GROUP_COUNT * GROUP_SIZE)
    for stream_data in vvdata.vtuber_videos[streamer]:
        for comment in stream_data.comments:
            if not_ignore_comment_data(comment) and comment.contentOffsetSeconds < MAX_GROUP_COUNT * GROUP_SIZE:
                comments_counts[comment.contentOffsetSeconds] += 1
    
    weighed_comments_counts = comments_counts / weights
    
    result = []
    for el in np.array_split(weighed_comments_counts, MAX_GROUP_COUNT):
        result.append(np.sum(el))
    result = np.array(result)

    return result

In [ ]:
VTUBER_NAMES = data.vtuber_names
LEN = len(VTUBER_NAMES)
vtuber_average = np.zeros(LEN)
vtuber_std = np.zeros(LEN)
for (i, vtuber) in zip(range(LEN), VTUBER_NAMES):
    chat_data = get_streamer_chat_data(vtuber)
    vtuber_average[i] = np.average(chat_data)
    vtuber_std[i] = np.std(chat_data)
df = pd.DataFrame({
    "Chat_Average": vtuber_average,
    "Chat_Std": vtuber_std,
    "Followers": list(map(lambda vtuber: len(data.vtuber_followers[vtuber]), VTUBER_NAMES)),
    "Followers_Error_x": np.ones(LEN),
    "Vtubers": VTUBER_NAMES
})

In [ ]:
fig = px.scatter(df, 
                 x="Followers",
                 y="Chat_Average",
                 error_x="Followers_Error_x",
                 error_y="Chat_Std",
                 hover_name="Vtubers")
fig.show()

In [ ]:
! pip install transformers
! pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126
! pip install wheel flash-attn

In [ ]:
import torch
from torch import Tensor
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer 

max_length = 512
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-large-instruct')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-large-instruct').cuda()

In [ ]:
def average_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

def embed(ids: List[str], input_texts: List[str]):
    all_loaded = True
    result = []
    for id in ids:
        cache_path = f"./data/.temp/message/{id}.json"

        if not os.path.exists(cache_path):
            all_loaded = False
            break
        else:
            with open(cache_path, 'r') as cache_file:
                result.append(json.load(cache_file))
    
    if all_loaded:
        print("[LOG] Loading from cache")
        
        return np.array(result)
        
    batch_dict = tokenizer(
        input_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    batch_dict.to(model.device)
    outputs = model(**batch_dict)
    embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

    # normalize embeddings
    embeddings = F.normalize(embeddings, p=2, dim=1)
    embeddings = embeddings.detach().cpu().numpy()

    print("[LOG]", "Caching result")
    for (id, id_embeddings) in zip(ids, embeddings.tolist()):
        cache_path = f"./data/.temp/message/{id}.json"
        with open(cache_path, 'w') as cache_file:
            json.dump(id_embeddings, cache_file, indent=4)

    return embeddings

In [ ]:
STREAMER = 'deme'
messages = []
for video_id in load_streamer_videos(STREAMER, log=True, cacheonly=False):
    video_data = load_video_data(video_id)
    for message_data in video_data.comments:
        messages.append(message_data)
len(messages)

In [ ]:
import re

def modify_string(st: str):
    curr_msg = st
    curr_msg = re.sub(r'@[\S]+', '', curr_msg)
    if len(curr_msg) > 200:
        curr_msg = curr_msg[:200]
    return curr_msg.strip()

dataset = list(map(lambda el: (el.message, el), messages))
dataset = list(map(lambda el: (el[0].as_plain_text, el[1]), dataset))
dataset = list(map(lambda el: (el[0].strip(), el[1]), dataset))
dataset = list(map(lambda el: (modify_string(el[0]), el[1]), dataset))
dataset = list(filter(lambda el: el[0] != '', dataset))
len(dataset)

In [ ]:
embeddings = []
step = 10
for i in range(0, len(dataset), step):
    print(f"{i}/{len(dataset)}")
    page = dataset[i:(i + step)]
    curr_msgs = list(map(lambda el: el[0], page))
    ids = list(map(lambda el: el[1].id, page))
    curr_embeddings = embed(ids, curr_msgs)
    embeddings.extend(curr_embeddings)

In [ ]:
embeddings = np.array(embeddings)
embeddings

In [ ]:
projection = sklearn.manifold.TSNE(n_components=2, verbose=1, max_iter=500, angle=0.85, random_state=1).fit_transform(embeddings)

In [ ]:
# labels = sklearn.cluster.HDBSCAN(min_cluster_size=5, max_cluster_size=4000, metric='cosine').fit_predict(X=embeddings)
labels = sklearn.cluster.KMeans(n_clusters=500, random_state=123).fit_predict(X=embeddings)
# labels = sklearn.cluster.DBSCAN(eps=0.9, min_samples=5, metric="cosine").fit_predict(X=embeddings)

In [ ]:
data_frame = pd.DataFrame({
    "x": projection[:, 0],
    "y": projection[:, 1],
    "Кластер": labels,
    "Текст": list(map(lambda comment_data: comment_data[1].message.as_text, dataset))
})
data_frame["Кластер"] = data_frame["Кластер"].astype(str) #convert to string

In [ ]:
fig = px.scatter(data_frame, 
                 x="x", 
                 y="y",
                 color="Кластер", 
                 hover_name="Текст")
fig.show()

## Посмотреть график количества фолловов среди вчуб среди чаччерсов

In [ ]:
STREAMER = "..."
chatters = []
chatter_ids = set()
for video_id in load_streamer_videos(STREAMER, log=True, cacheonly=True):
    video_data = load_video_data(video_id, log=True)
    for message_data in video_data.comments:
        if message_data.commenter is not None and message_data.commenter.id not in chatter_ids:
            chatter_ids.add(message_data.commenter.id)
            chatters.append(message_data.commenter)

In [ ]:
len(chatters), len(load_streamer_videos(STREAMER, log=False, cacheonly=True))

In [ ]:
df = pd.DataFrame({
    "Vtuber Followers": list(map(lambda chatter: min(0 if chatter.id not in data.user_following.keys() else len(data.user_following[chatter.id]), 200), chatters)),
    "Global Followers": list(map(lambda chatter: min(int(chatter.follows_count), 200), chatters)),
})

In [ ]:
fig = px.histogram(df, x="Vtuber Followers", nbins=201, range_x=(0, 200), histnorm="probability")
fig.show() 

In [ ]:
fig = px.histogram(df, x="Global Followers", nbins=201, range_x=(0, 200), histnorm="probability", range_y=(0, 0.06))
fig.show() 

In [ ]:
STREAMER = "nyamuras"
local_followers = np.array(list(map(lambda follower: 0 if follower.user is None else min(len(data.user_following[follower.user.id]), 200), data.vtuber_followers[STREAMER])))
global_followers = np.array(list(map(lambda follower: 0 if follower.user is None else min(follower.user.follows_count, 200), data.vtuber_followers[STREAMER])))
df = pd.DataFrame({
    "Vtuber Followers": local_followers,
    "Global Followers": global_followers
})
print(len(local_followers), np.average(local_followers), np.median(local_followers))
print(len(global_followers), np.average(global_followers), np.median(global_followers))
fig = px.histogram(df, x="Vtuber Followers", nbins=201, range_x=(0, 200), histnorm="probability", range_y=(0, 0.4))
# fig.write_image(f"{STREAMER}_local_follows.png", width=1980, height=1080)
fig.show() 
fig = px.histogram(df, x="Global Followers", nbins=201, range_x=(0, 200), histnorm="probability", range_y=(0, 0.05))
# fig.write_image(f"{STREAMER}_global_follows.png", width=1980, height=1080)
fig.show() 

## График “посещаемости” аудитории, то есть к какой процент стримов аудитория посещает

In [ ]:
STREAMER = "..."
chatters_appearence = {}
chatters = {}
for video_id in load_streamer_videos(STREAMER, log=True, cacheonly=True):
    video_data = load_video_data(video_id, log=True)
    chatter_ids = set()
    for message_data in video_data.comments:
        if message_data.commenter is not None and message_data.commenter.id not in chatter_ids:
            chatter_ids.add(message_data.commenter.id)
            chatters_appearence[message_data.commenter.id] = chatters_appearence.get(message_data.commenter.id, 0) + 1
            chatters[message_data.commenter.id] =  message_data.commenter

In [ ]:
df = pd.DataFrame({
    "Chat visits": list(chatters_appearence.values())
})
fig = px.histogram(df, x="Chat visits")
fig.show() 

## График локальных и глобальных фолловов у разных стримеров

In [ ]:
VTUBER_NAMES =  data.vtuber_names
LEN = len(VTUBER_NAMES)
global_followers_avg = np.zeros(LEN)
global_followers_std = np.zeros(LEN)
local_followers_avg = np.zeros(LEN)
local_followers_median = np.zeros(LEN)
local_followers_std = np.zeros(LEN)
for (i, vtuber) in zip(range(LEN), VTUBER_NAMES):
    followers = data.vtuber_followers[vtuber]
    dt = np.array(list(map(lambda follower: 0 if follower.user is None else len(data.user_following[follower.user.id]), followers)))
    local_followers_avg[i] = np.average(dt)
    local_followers_median[i] = np.median(dt)
    local_followers_std[i] = np.std(dt)
    dt = list(map(lambda follower: 0 if follower.user is None else follower.user.follows_count, followers))
    global_followers_avg[i] = np.median(dt)
    global_followers_std[i] = np.std(dt)
df = pd.DataFrame({
    "Global_Followers_Avg": global_followers_avg,
    "Global_Followers_Std": global_followers_std,
    "Local_Followers_Avg": local_followers_avg,
    "Local_Followers_Std": local_followers_std,
    "Local_Followers_Median": local_followers_median,
    "Local_Global_Followers_Delta": global_followers_avg - local_followers_avg,
    "Followers": list(map(lambda vtuber: len(data.vtuber_followers[vtuber]), VTUBER_NAMES)),
    "Followers_Error_x": np.ones(LEN),
    "Vtubers": VTUBER_NAMES,
    "color": local_followers_median > 11,
})
# fig = px.scatter(df, 
#                  x="Followers",
#                  y="Global_Followers_Avg",
#                 #  error_x="Followers_Error_x",
#                 #  error_y="Global_Followers_Std",
#                  hover_name="Vtubers")
# fig.show()
fig = px.scatter(df, 
                 x="Followers",
                 y="Local_Followers_Avg",
                #  error_x="Followers_Error_x",
                #  error_y="Local_Followers_Std",
                 hover_name="Vtubers")
# fig.write_image(f"vtubers_local_followers_avg.png", width=1980, height=1080)
fig.show()
# fig = px.scatter(df, 
#                  x="Followers",
#                  y="Local_Followers_Median",
#                 #  error_x="Followers_Error_x",
#                 #  error_y="Local_Followers_Std",
#                  hover_name="Vtubers",
#                  color="color")
# fig.show()
# fig = px.scatter(df, 
#                  x="Followers",
#                  y="Local_Global_Followers_Delta",
#                  hover_name="Vtubers")
# fig.show()

In [ ]:
np.percentile(local_followers_avg, 50)

In [ ]:
np.sum(local_followers_median > 11) / LEN

In [ ]:
fig = px.histogram(df["Local_Followers_Avg"], range_x=(0, 100), nbins=100, cumulative=True, histnorm="percent")
fig.show()

fig = px.histogram(df["Local_Followers_Median"], range_x=(0, 60), nbins=60, cumulative=True, histnorm="percent")
fig.show()

In [ ]:
user_first_follow = {}
for vtuber in data.vtuber_names:
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None:
            if follower.user.id not in user_first_follow.keys():
                user_first_follow[follower.user.id] = (follower.followed_at, vtuber)
            else:
                prev_followed_at, prev_vtuber = user_first_follow[follower.user.id]
                if prev_followed_at > follower.followed_at:
                    user_first_follow[follower.user.id] = (follower.followed_at, vtuber)

In [ ]:
vtuber_auditory = {}
for (user, (_, first_follow_vtuber)) in user_first_follow.items():
    vtuber_auditory[first_follow_vtuber] = vtuber_auditory.get(first_follow_vtuber, 0) + 1

In [ ]:
auditory_dataset = np.array([vtuber_auditory[vtuber] for vtuber in data.vtuber_names])
followers_dataset = np.array([len(data.vtuber_followers[vtuber]) for vtuber in data.vtuber_names])
auditory_percentage_dataset = auditory_dataset / followers_dataset * 100
vtuber_names_dataset = np.array(data.vtuber_names)
df = pd.DataFrame({
    "Auditory by vtuber": auditory_dataset,
    "Followers": followers_dataset,
    "Auditory by vtuber (%)": auditory_percentage_dataset,
    "Vtubers": vtuber_names_dataset
})
fig = px.scatter(df, 
                 x="Followers",
                 y="Auditory by vtuber (%)",
                 hover_name="Vtubers")
fig.show()

In [ ]:
strange_auditory = {}
for vtuber in data.vtuber_names:
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None:
            if vtuber not in strange_auditory.keys():
                strange_auditory[vtuber] = []
            if follower.user.follows_count in range(1900, 2100):
                strange_auditory[vtuber].append(follower)

In [ ]:
strange_auditory_dataset = np.array([len(strange_auditory[vtuber]) for vtuber in data.vtuber_names])
followers_dataset = np.array([len(data.vtuber_followers[vtuber]) for vtuber in data.vtuber_names])
vtuber_names_dataset = np.array(data.vtuber_names)
df = pd.DataFrame({
    "Strange auditory": strange_auditory_dataset,
    "Followers": followers_dataset,
    "Vtubers": vtuber_names_dataset
})
fig = px.scatter(df, 
                 x="Followers",
                 y="Strange auditory",
                 hover_name="Vtubers")
fig.show()

### Новореги

In [ ]:
PITOMNIK = ['sleduck', 'xkamysh', 'nyamuras', 'blin_ya', 'squimeh', 'simfonira', 'maroosha', 'solemuri', 'd_maf_', 'rera_seal', 'miuchia',
            'yuislime', 'qchaan_9', 'banshameow', 'ellysteiny']

novoregs_map = dict()

def is_novoreg(follower_data: FollowerData):
    if follower_data.user is not None:
        local_follows = len(data.user_following[follower_data.user.id])
        global_follows = follower_data.user.follows_count
        pitomnik_follows = len(list(filter(lambda el: el in PITOMNIK, data.user_following[follower_data.user.id])))

        # return pitomnik_follows >= 0.8 * global_follows \
        #     and (follower_data.followed_at - follower_data.user.created_at) < timedelta(days=7)
        return pitomnik_follows >= 0.5 * global_follows or global_follows < 4
    else:
        return False

for vtuber in PITOMNIK:
    print(vtuber)
    novoregs = 0
    
    for follower_data in data.vtuber_followers[vtuber]:
        if is_novoreg(follower_data):
            novoregs += 1
            novoregs_map[follower_data.user.id] = novoregs_map.get(follower_data.user.id, []) + [vtuber]
            # print(novoregs, follower_data)
    print("====", novoregs, "====")

In [ ]:
len(novoregs_map), len(list(filter(lambda el: el[1] == ['xkamysh'], novoregs_map.items())))

In [ ]:
list(filter(lambda el: len(el[1]) > 2, novoregs_map.items()))